In [ ]:
!pip install -q google-genai faiss-cpu

In [ ]:
from google import genai
import faiss
import numpy as np
import json

client = genai.Client(api_key="API_KEY_HERE")

In [ ]:
# Load the FAISS database

index = faiss.read_index(
    "places_faiss.index"
)

print("Number of vectors:", index.ntotal)

Number of vectors: 46


In [ ]:
# Load chunk metadata

with open("chunks_metadata.json", "r", encoding="utf-8") as file:

    all_chunks = json.load(file)

print("Number of chunks:", len(all_chunks))

Number of chunks: 46


In [ ]:
# Define the RAG function

def ask_question(question, top_k=3):

    print("\n================================")
    print("QUESTION")
    print("================================")
    print(question)

    # Convert question into embedding

    query_result = client.models.embed_content(
        model="gemini-embedding-2",
        contents=question
    )

    query_vector = np.array(
        [query_result.embeddings[0].values],
        dtype="float32"
    )

    # Normalize query vector

    faiss.normalize_L2(query_vector)

    # Search the vector database

    scores, indices = index.search(
        query_vector,
        top_k
    )

    print("\n================================")
    print("TOP RELEVANT CHUNKS")
    print("================================")

    retrieved_chunks = []
    references = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), 1
    ):

        chunk = all_chunks[idx]

        print(f"\n--- Result {rank} ---")
        print(f"Cosine Similarity: {score:.4f}")
        print(f"Source: {chunk['file_name']}")
        print("Text:")
        print(chunk["text"])

        retrieved_chunks.append(chunk["text"])

        if chunk["file_name"] not in references:
            references.append(chunk["file_name"])

    # Combine retrieved chunks

    context = "\n\n".join(
        retrieved_chunks
    )

    # Build the LLM prompt

    prompt = f"""
You are a question-answering assistant.

Answer the user's question using ONLY the provided context.

Context:
{context}

User Question:
{question}

Instructions:
- Use only the information in the context.
- Do not add information from outside the context.
- Give the answer directly and briefly.
- If the context does not contain enough information,
  say: "I don't have enough information to answer this."
"""

    # Generate the final answer

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    # Print final output

    print("\n================================")
    print("FINAL OUTPUT")
    print("================================")

    print("\nUser Question:")
    print(question)

    print("\nGenerated Answer:")
    print(response.text)

    print("\nReferences:")

    for reference in references:
        print(f"- {reference}")

    return response.text

In [ ]:
# Test a city question

answer = ask_question(
    "What is AlUla famous for?"
)


QUESTION
What is AlUla famous for?

TOP RELEVANT CHUNKS

--- Result 1 ---
Cosine Similarity: 0.8017
Source: Saudi Tourism.txt
Text:
AlUla also offers hiking, stargazing, hot-air balloon experiences, cultural events, and modern attractions such as Maraya. The destination is particularly attractive to visitors interested in archaeology, history, nature, photography, and cultural tourism. 2. Diriyah: The Birthplace of the First Saudi State Diriyah is a historic destination located on the outskirts of Riyadh and is strongly connected to the history and identity of Saudi Arabia. It was the capital of the First Saudi State and is home to At-Turaif District, a UNESCO World Heritage Site known for its traditional Najdi mud-brick architecture.

--- Result 2 ---
Cosine Similarity: 0.7887
Source: Saudi Tourism.txt
Text:
Saudi Tourism 1. AlUla: Ancient Heritage and Desert Landscapes AlUla is one of Saudi Arabia’s most distinctive tourism destinations, combining ancient heritage, dramatic desert l

In [ ]:
# Test a project question

answer = ask_question(
    "Which project focuses on entertainment, sports, culture, and tourism?"
)


QUESTION
Which project focuses on entertainment, sports, culture, and tourism?

TOP RELEVANT CHUNKS

--- Result 1 ---
Cosine Similarity: 0.7535
Source: Saudi Projects & Vision 2030.txt
Text:
The Line therefore represents the combination of architecture, technology, transportation, environmental planning, and economic development within Vision 2030. 3. Qiddiya: Entertainment, Sports, and Culture Qiddiya is a major development near Riyadh focused on entertainment, sports, culture, and tourism. The project aims to create a large destination where residents and visitors can participate in recreational, sporting, cultural, and entertainment activities. Qiddiya is designed to include attractions and facilities for different age groups and interests, including sports venues, theme parks, cultural experiences, and outdoor activities.

--- Result 2 ---
Cosine Similarity: 0.7257
Source: Saudi Projects & Vision 2030.txt
Text:
The development is part of Saudi Arabia’s efforts to expand domestic e